# Zomato Bangalore Restaurants — EDA & Feature Engineering

Exploratory data analysis and feature engineering on the **Zomato Bangalore Restaurants** dataset (51,717 restaurants, 17 columns). This notebook covers data cleaning, missing value handling, categorical encoding, and a visual analysis of the factors associated with restaurant ratings.

**Dataset:** [Zomato Bangalore Restaurants](https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants)

**GitHub repo:** [nadiucar/zomato-bangalore-eda-encoding](https://github.com/nadiucar/zomato-bangalore-eda-encoding)

**Contents**
1. [Setup & Data Loading](#setup)
2. [Initial Exploration](#explore)
3. [Data Cleaning](#cleaning)
4. [Missing Value Handling](#missing)
5. [Encoding](#encoding)
6. [Exploratory Data Analysis](#eda)
7. [Key Findings](#findings)

## 1. Setup & Data Loading <a id="setup"></a>

If you're running this on Kaggle, attach the dataset via **Add Input** (search for "Zomato Bangalore Restaurants") and adjust the path below to match the input folder Kaggle assigns.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Adjust this path to match your Kaggle input folder
df = pd.read_csv("/kaggle/input/zomato-bangalore-restaurants/zomato.csv")
df.head()

## 2. Initial Exploration <a id="explore"></a>

Before touching anything, let's understand the shape, dtypes, and missingness of the raw data.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['rate'].unique()

`rate` contains inconsistent values like `"4.1/5"`, `"NEW"`, and `"-"` — this needs cleaning before it can be used as a numeric column.

## 3. Data Cleaning <a id="cleaning"></a>

We work on a copy of the raw data (`df_clean`) so the original stays untouched for reference.

In [ ]:
df_clean = df.copy()

### 3.1 Rating column (`rate` → `rate(over 5)`)

- Replace placeholder values (`"NEW"`, `"-"`) with `NaN`
- Strip the `/5` suffix
- Convert to `float`
- Rename for clarity

This raised the missing count from 7,775 to 10,052 — placeholder values like `"NEW"` and `"-"` were hiding additional missing data that a plain `isnull()` check couldn't catch.

In [ ]:
df_clean['rate'] = df_clean['rate'].replace("NEW", np.nan)
df_clean['rate'] = df_clean['rate'].replace("-", np.nan)
df_clean['rate'] = df_clean['rate'].str.replace("/5", "")
df_clean['rate'] = df_clean['rate'].astype(float)

df_clean.rename(columns={'rate': 'rate(over 5)'}, inplace=True)
df_clean['rate(over 5)'].describe()

### 3.2 Phone column

Some restaurants list two numbers on the same row, separated by `\r\n`. We split this into `phone_first` and `phone_secondary`, then strip `+` and spaces for a consistent format.

In [ ]:
df_clean[['phone_first', 'phone_secondary']] = df_clean['phone'].str.split(r'\r?\n', n=1, expand=True)

chars_to_remove = ["+", " "]
cols_to_clean = ["phone_first", "phone_secondary"]

for item in chars_to_remove:
    for col in cols_to_clean:
        df_clean[col] = df_clean[col].str.replace(item, "")

df_clean = df_clean.drop(columns="phone")
df_clean[['phone_first', 'phone_secondary']].head()

### 3.3 Cost column (`approx_cost(for two people)`)

Remove thousands separators (`,`) and convert to `float`.

In [ ]:
df_clean['approx_cost(for two people)'] = df_clean['approx_cost(for two people)'].str.replace(",", "")
df_clean['approx_cost(for two people)'] = df_clean['approx_cost(for two people)'].astype(float)

df_clean['approx_cost(for two people)'].describe()

## 4. Missing Value Handling <a id="missing"></a>

Rather than applying one blanket strategy, each column is handled according to what actually makes sense for it — group-based imputation where a meaningful group exists, mode imputation for low-missingness categoricals, and dropping where the column itself is unreliable.

### 4.1 `rate(over 5)` — group median by `listed_in(type)`

Restaurants of the same service type (Delivery, Dine-out, Cafes, ...) tend to have similar rating patterns, so we fill missing ratings with their own group's median rather than a single global value.

In [ ]:
df_clean.groupby('listed_in(type)')['rate(over 5)'].median()

In [ ]:
df_clean['rate(over 5)'] = df_clean.groupby('listed_in(type)')['rate(over 5)'].transform(lambda x: x.fillna(x.median()))

### 4.2 `dish_liked` — dropped

~54% of this column is missing. At that level, imputing it would make the column unreliable for analysis, so it's dropped entirely.

In [ ]:
df_clean = df_clean.drop(columns="dish_liked")

### 4.3 `menu_item` — dropped

This column isn't `NaN` for empty entries — it stores an empty list as the literal string `'[]'`, which `isnull()` doesn't catch. Checking directly:

In [ ]:
(df_clean['menu_item'] == '[]').sum() / len(df_clean) * 100

~76% of rows have no menu data. Dropped for the same reason as `dish_liked`.

In [ ]:
df_clean = df_clean.drop(columns="menu_item")

### 4.4 `location` / `cuisines` — rows dropped

Only 21 and 45 missing values respectively — a negligible share of 51,717 rows, so the corresponding rows are simply dropped rather than imputed.

In [ ]:
drop_na_col = ['location', 'cuisines']
df_clean = df_clean.dropna(subset=drop_na_col)

### 4.5 `rest_type` — mode imputation

227 missing values in a categorical column. Filled with the most frequent category (`Quick Bites`, 19,335 occurrences).

In [ ]:
df_clean['rest_type'] = df_clean['rest_type'].fillna(df_clean['rest_type'].mode()[0])

### 4.6 `approx_cost(for two people)` — group median by `listed_in(type)`

Same logic as the rating column — restaurants of the same service type tend to have similar price points.

In [ ]:
df_clean['approx_cost(for two people)'] = df_clean.groupby('listed_in(type)')['approx_cost(for two people)'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df_clean.isnull().sum()

`phone_first` / `phone_secondary` are the only columns left with missing values — and that's expected: most restaurants simply only have one phone number listed, so `phone_secondary` being empty is structural, not a data quality issue.

## 5. Encoding <a id="encoding"></a>

One-hot encoding for the low-cardinality categorical columns, with `drop_first=True` to avoid the dummy variable trap.

In [ ]:
df_clean = pd.get_dummies(df_clean, columns=['online_order', 'book_table'], drop_first=True)
df_clean = pd.get_dummies(df_clean, columns=['listed_in(type)'], drop_first=True)

df_clean.info()

> `book_table` and `online_order` each collapse to a single `_Yes` column. For `listed_in(type)`, the alphabetically-first category (`Buffet`) is the dropped reference — rows where every `listed_in(type)_*` column is `False` belong to `Buffet`.

## 6. Exploratory Data Analysis <a id="eda"></a>

### 6.1 Correlation matrix

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(df_clean.corr(numeric_only=True), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

**Reading the matrix:**
- `book_table` and `approx_cost` show the strongest relationship (**0.62**) — restaurants that accept table reservations tend to be significantly more expensive.
- `rate` correlates moderately with `votes` (0.42), `book_table` (0.41), and `approx_cost` (0.37).
- The negative correlations among `listed_in(type)_*` dummy columns (e.g. Delivery vs. Dine-out: -0.73) are a **structural artifact of one-hot encoding** — a restaurant can only belong to one service type, so these columns are mechanically inverse to each other. Not a real-world relationship.

### 6.2 Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_clean['rate(over 5)'], kde=True, ax=axes[0])
axes[0].set_title('Rate Distribution')

sns.histplot(df_clean['votes'], kde=True, ax=axes[1])
axes[1].set_title('Votes Distribution')

sns.histplot(df_clean['approx_cost(for two people)'], kde=True, ax=axes[2])
axes[2].set_title('Approx Cost Distribution')

plt.tight_layout()
plt.show()

**Reading the distributions:**
- `votes` and `approx_cost` are strongly right-skewed (long-tail distributions) — most restaurants cluster at low values, with a handful of very popular / very expensive outliers stretching the tail.
- `rate(over 5)` shows an artificial spike around **3.7**. This traces back to the group-median imputation in section 4.1: `Delivery` and `Dine-out` together make up 84% of the dataset, and both happen to share a median rating of 3.7 — so thousands of imputed values landed on the exact same point. This is a **methodological artifact of imputation**, not an organic rating pattern, and is worth calling out explicitly in any downstream analysis.

### 6.3 Cost vs. table booking

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='book_table_Yes', y='approx_cost(for two people)', data=df_clean)
plt.title('Cost by Table Booking Availability')
plt.tight_layout()
plt.show()

Median cost without table booking is roughly **₹400–450**, versus roughly **₹1,200** with table booking — about 3x higher, with almost no overlap between the two groups' interquartile ranges. Table reservations are strongly associated with the higher-end restaurant segment.

### 6.4 Rate vs. votes / cost / table booking

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.scatterplot(data=df_clean, x='votes', y='rate(over 5)', alpha=0.3, ax=axes[0])
axes[0].set_title('Votes vs Rate')

sns.scatterplot(data=df_clean, x='approx_cost(for two people)', y='rate(over 5)', alpha=0.3, ax=axes[1])
axes[1].set_title('Approx Cost vs Rate')

sns.boxplot(data=df_clean, x='book_table_Yes', y='rate(over 5)', ax=axes[2])
axes[2].set_title('Book Table vs Rate')

plt.tight_layout()
plt.show()

**Reading these three panels:**
- **Votes vs. Rate:** restaurants with few votes show wide rating variance (as low as 1.8, as high as 4.9); restaurants with many votes converge toward 4.0–4.9. This is a classic small-sample variance effect — a rating based on a handful of votes is far less stable than one based on thousands.
- **Cost vs. Rate:** a milder version of the same funnel shape — higher-cost restaurants are less likely to have very low ratings, but the effect is weaker than for votes.
- **Book Table vs. Rate:** restaurants that accept table bookings have a visibly higher and tighter rating distribution (median ~4.2) than those that don't (median ~3.7), with almost no overlap in interquartile range.

## 7. Key Findings <a id="findings"></a>

- **Table booking is the strongest, most consistent signal** in this dataset — it's associated with both higher cost (0.62 correlation) and higher, more tightly-distributed ratings.
- **Vote count matters for reliability, not just popularity** — low-vote restaurants show much wider rating variance, consistent with small-sample statistics rather than a real quality difference.
- **The 3.7 spike in the rating distribution is an imputation artifact**, not a genuine pattern — worth flagging in any report or model built on this data.
- **`dish_liked` and `menu_item` were too sparse to use** (54% and 76% missing respectively) and were dropped rather than imputed, to avoid injecting unreliable signal.

### Possible next steps
- Multi-label encode `rest_type` and `cuisines` (they contain comma-separated multi-category values, e.g. `"Cafe, Casual Dining"`)
- Log-transform `votes` and `approx_cost` to reduce skew before modeling
- Build a regression/classification model to predict `rate(over 5)` using the engineered features above